In [1]:
import os
import sys 
sys.path.append('..')
# sys.path.append('../..')

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import chatgenie as cg 

ImportError: sklearn not installed , Please install scikit-learn


In [3]:
openai = cg.LLM(llm_type='openai', api_key=os.getenv('OPENAI_API_KEY'))
# embeder = cg.Embedder(embedder_type='openai', api_key=os.getenv('OPENAI_API_KEY'), model='text-davinci-003', dimesion=os.getenv('MONGO_DB_DIMENSION'))
# vectordb = cg.VectorDB(
#     db_type="mongo",
#     username=os.getenv('MONGODB_USERNAME'),
#     password=os.getenv('MONGODB_PASSWORD'),
#     dbname=os.getenv('MONGODB_DATABASE'),
#     collection_name=os.getenv('MONGODB_COLLECTION'),
#     dimensions=os.getenv('MONGODB_DIMENSION')
# )

/home/pramodyasahan/Github/chatgenie-library/chatgenie/llm/interface.py:18: UserWarning: Parameters {'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  self._cls_concrete = OpenAILlm(config)


## Testing Generator

In [5]:
DEFAULT_PROMPT = """
  Use the following pieces of context to answer the query at the end.

  $context

  Query: $query

  Helpful Answer:
""" 

DEFAULT_PROMPT_WITH_HISTORY = """
  Use the following pieces of context to answer the query at the end.
  If the question is not related to obesity or weight-loss, just say that you don't know, don't try to make up an answer.
  I will provide you with our conversation history.

  $context

  History: $history

  Query: $query

  Helpful Answer:
"""  

DOCS_SITE_DEFAULT_PROMPT = """
  Use the following pieces of context to answer the query at the end.
  If you don't know the answer, just say that you don't know, don't try to make up an answer. Wherever possible, give complete code snippet. Dont make up any code snippet on your own.

  $context

  Query: $query

  Helpful Answer:
"""  

In [6]:
generator = cg.Generator(llm=openai,
                         prompt_template=DEFAULT_PROMPT,
                         memory_type='buffer',
                         system_prompt="You are an AI assistant helping a user with their query.",
                         use_cache=True,)


/home/pramodyasahan/Github/chatgenie-library/chatgenie/agent/generator.py:29: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  return ConversationBufferMemory()


In [7]:
import time

start = time.time()
response1 = generator.generate(input_query="What is the meaning of life?")
end = time.time()
print(f"First call duration: {end - start} seconds")

start = time.time()
response2 = generator.generate(input_query="What is the meaning of life?")
end = time.time()
print(f"Second call duration: {end - start} seconds")


First call duration: 1.9948792457580566 seconds
Second call duration: 0.0005447864532470703 seconds


In [8]:
generator.generate(input_query="What is the meaning of life?")

'The meaning of life is a philosophical question that has been debated for centuries. Different people and cultures have different perspectives on this question. Some believe that the meaning of life is to seek happiness and fulfillment, while others think it is about spiritual growth or serving a higher purpose. Ultimately, the meaning of life is a deeply personal and subjective concept that each individual must explore and define for themselves.'

In [7]:
generator.generate_with_context(input_query="What is the meaning of life?", context="There is no meaning to life.")

'Based on the context provided that "There is no meaning to life," it can be interpreted that the concept of a universal or inherent meaning to life may not exist. Therefore, the meaning of life could be subjective and open to individual interpretation and personal beliefs. It could be about finding purpose, happiness, fulfillment, or making a positive impact in the world based on one\'s own values and goals.'

In [8]:
generator = cg.Generator(llm=openai,
                         prompt_template=DEFAULT_PROMPT_WITH_HISTORY,
                         memory_type='buffer',
                         system_prompt="You are an three word max answer generation agent.") 

g:\IXDLabs\Chat bot project[Halt]\chatgenie-library\notebooks\..\chatgenie\agent\generator.py:22: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  return ConversationBufferMemory()


In [9]:
generator.generate(input_query="Where is the Themes river located?")


"I don't know."

In [10]:
generator.generate_with_context(input_query="Where is the Themes river located?", context="The Themes river is located in London.")

'London, United Kingdom.'

In [11]:
generator.generate(input_query="Where is the Themes river located?")

'London, United Kingdom.'

## Judge Testing

In [13]:
from typing import Optional
from pydantic import BaseModel, Field

class Joke(BaseModel):
    """Joke to tell user."""

    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline to the joke")
    rating: Optional[int] = Field(
        default=None, description="How funny the joke is, from 1 to 10"
    )

judge = cg.Judge(llm=openai, rule=Joke)

In [14]:
judge.judge(prompt="give me a chicken joke")

c:\Users\nipun_qk4hy9e\miniconda3\envs\chatgenie_env\lib\site-packages\langchain_openai\chat_models\base.py:1390: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


Joke(setup='Why did the chicken join a band?', punchline='Because it had the drumsticks!', rating=8)

## Loader Testing 

In [4]:
loader = cg.Loader(llm=openai, embedder=embeder, db=vectordb)

In [5]:
loader.simple_load(source="../data/Easy_recipes.pdf")

ic| len(documents[0]["text_embedding"]): 1536
ic| self.config.dimensions: 1536


## Retriever Testing

In [4]:
retriever = cg.Retriever(embedder=embeder, db=vectordb, llm=openai)

In [ ]:
retriever.simple_retrieve("Give me a recipe name for dishes with meat")